# 🚁 Dron İzləmə Sistemi — OSTrack

**OSTrack** (One-Stream Tracking, ECCV 2022) modeli ilə dron videolarında tək obyekt izləmə.

### İstifadə qaydası:
1. **Runtime → Change runtime type → T4 GPU** seçin
2. Hüceyrələri ardıcıl icra edin (`Ctrl+F9`)
3. `⚙️ USER CONFIG` hüceyrəsini öz parametrlərinizlə doldurun
4. Son hüceyrə nəticə videosunu avtomatik yükləyəcək

---

In [ ]:
# ✅ 1. GPU Yoxlaması
import torch

if torch.cuda.is_available():
    print(f'✅ GPU aktiv: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   CUDA: {torch.version.cuda}')
else:
    print('❌ GPU tapılmadı! Runtime → Change runtime type → T4 GPU seçin.')


✅ GPU aktiv: Tesla T4
   VRAM: 15.6 GB
   CUDA: 12.8


In [ ]:
# 📦 2. OSTrack Quraşdırılması + Yamaqlar
import os, sys, subprocess, types, torch

# ── Repo ────────────────────────────────────────────────────────────────────
if not os.path.exists('/content/OSTrack'):
    print('OSTrack repo clonlanır...')
    subprocess.run(['git', 'clone', 'https://github.com/botaoye/OSTrack.git'], check=True)
    print('✅ Clone tamamlandı!')
else:
    print('✅ OSTrack artıq mövcuddur.')

os.chdir('/content/OSTrack')
if '/content/OSTrack' not in sys.path:
    sys.path.insert(0, '/content/OSTrack')

# ── Asılılıqlar ─────────────────────────────────────────────────────────────
print('Asılılıqlar quraşdırılır...')
os.system('apt-get install -qq libturbojpeg')
os.system('pip install -q timm==0.5.4 einops jpeg4py lmdb visdom wandb')
os.system('pip install -q -r requirements.txt 2>/dev/null || true')
print('✅ Quraşdırma tamamlandı!')

# ── torch._six yamağı (PyTorch 2.x-də silindi) ──────────────────────────────
if 'torch._six' not in sys.modules:
    _six = types.ModuleType('torch._six')
    _six.string_classes = (str, bytes)
    sys.modules['torch._six'] = _six
    torch._six = _six
    print('✅ torch._six yamağı tətbiq edildi!')

# ── loader.py yamağı ────────────────────────────────────────────────────────
loader_path = '/content/OSTrack/lib/train/data/loader.py'
with open(loader_path, 'r') as f:
    content = f.read()
if 'from torch._six import string_classes' in content:
    with open(loader_path, 'w') as f:
        f.write(content.replace(
            'from torch._six import string_classes',
            'string_classes = (str, bytes)'
        ))
    print('✅ loader.py yamandı!')

# ── basetracker.py yamağı (visdom opsional) ──────────────────────────────────
bt_path = '/content/OSTrack/lib/test/tracker/basetracker.py'
with open(bt_path, 'r') as f:
    content = f.read()
if 'from lib.vis.visdom_cus import Visdom' in content:
    with open(bt_path, 'w') as f:
        f.write(content.replace(
            'from lib.vis.visdom_cus import Visdom',
            'try:\n    from lib.vis.visdom_cus import Visdom\nexcept ImportError:\n    Visdom = None'
        ))
    print('✅ basetracker.py yamandı!')

# ── local.py ────────────────────────────────────────────────────────────────
os.makedirs('/content/OSTrack/lib/test/evaluation', exist_ok=True)
os.makedirs('/content/OSTrack/output/checkpoints/train/ostrack/vitb_256_mae_ce_32x4_ep300', exist_ok=True)
os.makedirs('/content/OSTrack/pretrained_networks', exist_ok=True)

with open('/content/OSTrack/lib/test/evaluation/local.py', 'w') as f:
    f.write("""class EnvironmentSettings:
    def __init__(self):
        self.workspace_dir       = '/content/OSTrack'
        self.tensorboard_dir     = '/content/OSTrack/tensorboard'
        self.pretrained_networks = '/content/OSTrack/pretrained_networks'
        self.save_dir            = '/content/OSTrack/output'
        self.prj_dir             = '/content/OSTrack'
        self.lasot_dir           = ''
        self.got10k_dir          = ''
        self.trackingnet_dir     = ''
        self.coco_dir            = ''
        self.lvis_dir            = ''
        self.tnl2k_dir           = ''
        self.isbi_dir            = ''
        self.nfs_dir             = ''
        self.uav_dir             = ''

def local_env_settings():
    return EnvironmentSettings()
""")
print('✅ Mühit konfiqurasiyası hazırdır!')


OSTrack repo clonlanır...
✅ Clone tamamlandı!
Asılılıqlar quraşdırılır...
✅ Quraşdırma tamamlandı!
✅ torch._six yamağı tətbiq edildi!
✅ loader.py yamandı!
✅ basetracker.py yamandı!
✅ Mühit konfiqurasiyası hazırdır!


In [ ]:
# 📥 3. Model Çəkilərini Yükləyin
import os, shutil, glob
os.system('pip install -q gdown')
import gdown

CKPT_DIR  = '/content/OSTrack/output/checkpoints/train/ostrack/vitb_256_mae_ce_32x4_ep300'
CKPT_PATH = f'{CKPT_DIR}/OSTrack_ep0300.pth.tar'
os.makedirs(CKPT_DIR, exist_ok=True)

if os.path.exists(CKPT_PATH) and os.path.getsize(CKPT_PATH) > 1e6:
    print(f'✅ Model artıq mövcuddur ({os.path.getsize(CKPT_PATH)/1e6:.0f} MB)')
else:
    FOLDER_ID = '1ttafo0O5S9DXK2PX0YqPvPrQ-HWJjhSy'
    TMP_DIR   = '/tmp/ostrack_ckpts'
    print(f'Model yüklənir: https://drive.google.com/drive/folders/{FOLDER_ID}\n')
    try:
        gdown.download_folder(id=FOLDER_ID, output=TMP_DIR, quiet=False, use_cookies=False)
        found = (glob.glob(f'{TMP_DIR}/*256*/*ep0300*', recursive=True) or
                 glob.glob(f'{TMP_DIR}/**/*ep0300*.pth.tar', recursive=True))
        if not found:
            all_files = glob.glob(f'{TMP_DIR}/**/*.pth.tar', recursive=True)
            found = [next((f for f in all_files if '256' in f), all_files[0])] if all_files else []
        if found:
            shutil.copy(found[0], CKPT_PATH)
            print(f'✅ Kopyalandı: {CKPT_PATH}')
    except Exception as e:
        print(f'❌ Yükləmə xətası: {e}')
        print('Manual həll: https://drive.google.com/drive/folders/1ttafo0O5S9DXK2PX0YqPvPrQ-HWJjhSy')
        print('vitb_256_mae_ce_32x4_ep300/OSTrack_ep0300.pth.tar faylını Colab-a upload edin.')
        manual = glob.glob('/content/OSTrack_ep0300.pth.tar')
        if manual:
            shutil.copy(manual[0], CKPT_PATH)
            print('✅ Manual upload tapıldı!')

if os.path.exists(CKPT_PATH) and os.path.getsize(CKPT_PATH) > 1e6:
    print(f'✅ Model hazırdır: {os.path.getsize(CKPT_PATH)/1e6:.0f} MB')
else:
    print('❌ Model tapılmadı. Yuxarıdakı manual həlldən istifadə edin.')


Model yüklənir: https://drive.google.com/drive/folders/1ttafo0O5S9DXK2PX0YqPvPrQ-HWJjhSy



Retrieving folder contents


Retrieving folder 1nU40_fpIVUd4ht3T_qB8uSBhQeSzBTD9 vitb_256_mae_32x4_ep300
Processing file 1lpmc5DhZTIluKdvawvt-5iOpLZtcRjm4 OSTrack_ep0300.pth.tar
Retrieving folder 1H4Fb7t3kXp7P8IDf5FLN-OaO9kXf25WR vitb_256_mae_ce_32x4_ep300
Processing file 1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV OSTrack_ep0300.pth.tar
Retrieving folder 1NEvJe8AW36tt2-jbwpeOJxH-KP4t9JKm vitb_256_mae_ce_32x4_got10k_ep100
Processing file 1jwTLPiwj_r7KkAYIFdH9lkbt1JH5MhGC OSTrack_ep0100.pth.tar
Retrieving folder 1ov2nU3itkR5FJBn_wiapMGxpsL6aB5f6 vitb_384_mae_32x4_ep300
Processing file 1rTD1W_gr1VNsPg-rjQHN5t2VfI7_Kk3n OSTrack_ep0300.pth.tar
Retrieving folder 1XJ70dYB6muatZ1LPQGEhyvouX-sU_wnu vitb_384_mae_ce_32x4_ep300
Processing file 19tj5NmwZr9ylDaqOwfNVOLijr389ZDBi OSTrack_ep0300.pth.tar
Retrieving folder 1Aa2ChMp4T3vAphN8RvI_Q3CWyUkL-yyU vitb_384_mae_ce_32x4_got10k_ep100
Processing file 1WKaXPz1l3SdQWIFzq18ByrdxFJXtHMzV OSTrack_ep0100.pth.tar


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1lpmc5DhZTIluKdvawvt-5iOpLZtcRjm4
From (redirected): https://drive.google.com/uc?id=1lpmc5DhZTIluKdvawvt-5iOpLZtcRjm4&confirm=t&uuid=7eea0347-a1f8-415d-8b67-9b897c76f517
To: /tmp/ostrack_ckpts/vitb_256_mae_32x4_ep300/OSTrack_ep0300.pth.tar
100%|██████████| 370M/370M [00:06<00:00, 58.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV
From (redirected): https://drive.google.com/uc?id=1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV&confirm=t&uuid=5000fe4b-b2a3-4d5c-8cae-e0ac4a1c7557
To: /tmp/ostrack_ckpts/vitb_256_mae_ce_32x4_ep300/OSTrack_ep0300.pth.tar
100%|██████████| 370M/370M [00:06<00:00, 53.8MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1jwTLPiwj_r7KkAYIFdH9lkbt1JH5MhGC
From (redirected): https://drive.google.com/uc?id=1jwTLPiwj_r7KkAYIFdH9lkbt1JH5MhGC&con

✅ Kopyalandı: /content/OSTrack/output/checkpoints/train/ostrack/vitb_256_mae_ce_32x4_ep300/OSTrack_ep0300.pth.tar
✅ Model hazırdır: 370 MB


In [ ]:
# ⚙️ 4. USER CONFIG — Yalnız bu hüceyrəni redaktə edin!
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

VIDEO_PATH  = '/content/moving_porche.mp4'   # ← Video faylının yolu
OUTPUT_PATH = '/content/tracked_output.mp4'  # ← Çıxış video yolu
INIT_BBOX   = [1500, 1200, 400, 500]          # ← [x, y, en, hündürlük] (piksel)

BOX_COLOR     = (0, 255, 0)  # Bounding box rəngi (B, G, R)
BOX_THICKNESS = 2
SHOW_TRAIL    = True          # İz göstərilsin?
TRAIL_LENGTH  = 30            # Neçə frame-lik iz
MODEL_NAME    = 'vitb_256_mae_ce_32x4_ep300'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('✅ Konfiqurasiya:')
print(f'   Video  : {VIDEO_PATH}')
print(f'   Çıxış  : {OUTPUT_PATH}')
print(f'   BBox   : x={INIT_BBOX[0]}, y={INIT_BBOX[1]}, w={INIT_BBOX[2]}, h={INIT_BBOX[3]}')
print(f'   Model  : {MODEL_NAME}')


✅ Konfiqurasiya:
   Video  : /content/moving_porche.mp4
   Çıxış  : /content/tracked_output.mp4
   BBox   : x=1500, y=1200, w=400, h=500
   Model  : vitb_256_mae_ce_32x4_ep300


In [ ]:
# 🤖 5. Tracker Yüklənməsi
import importlib

CKPT_PATH = (f'/content/OSTrack/output/checkpoints/train/ostrack/'
             f'{MODEL_NAME}/OSTrack_ep0300.pth.tar')

print('Tracker yüklənir...')
param_module = importlib.import_module('lib.test.parameter.ostrack')
params = param_module.parameters(MODEL_NAME)
params.checkpoint = CKPT_PATH
params.debug = 0

from lib.test.tracker.ostrack import OSTrack
tracker = OSTrack(params, 'lasot')

print(f'✅ Tracker yükləndi!')
print(f'   Model      : {MODEL_NAME}')
print(f'   Checkpoint : {CKPT_PATH}')


Tracker yüklənir...
test config:  {'MODEL': {'PRETRAIN_FILE': 'mae_pretrain_vit_base.pth', 'EXTRA_MERGER': False, 'RETURN_INTER': False, 'RETURN_STAGES': [], 'BACKBONE': {'TYPE': 'vit_base_patch16_224_ce', 'STRIDE': 16, 'MID_PE': False, 'SEP_SEG': False, 'CAT_MODE': 'direct', 'MERGE_LAYER': 0, 'ADD_CLS_TOKEN': False, 'CLS_TOKEN_USE_MODE': 'ignore', 'CE_LOC': [3, 6, 9], 'CE_KEEP_RATIO': [0.7, 0.7, 0.7], 'CE_TEMPLATE_RANGE': 'CTR_POINT'}, 'HEAD': {'TYPE': 'CENTER', 'NUM_CHANNELS': 256}}, 'TRAIN': {'LR': 0.0004, 'WEIGHT_DECAY': 0.0001, 'EPOCH': 300, 'LR_DROP_EPOCH': 240, 'BATCH_SIZE': 32, 'NUM_WORKER': 10, 'OPTIMIZER': 'ADAMW', 'BACKBONE_MULTIPLIER': 0.1, 'GIOU_WEIGHT': 2.0, 'L1_WEIGHT': 5.0, 'FREEZE_LAYERS': [0], 'PRINT_INTERVAL': 50, 'VAL_EPOCH_INTERVAL': 20, 'GRAD_CLIP_NORM': 0.1, 'AMP': False, 'CE_START_EPOCH': 20, 'CE_WARM_EPOCH': 80, 'DROP_PATH_RATE': 0.1, 'SCHEDULER': {'TYPE': 'step', 'DECAY_RATE': 0.1}}, 'DATA': {'SAMPLER_MODE': 'causal', 'MEAN': [0.485, 0.456, 0.406], 'STD': [0.2

In [ ]:
# 🎬 6. İzləmə Prosesi
import cv2, numpy as np, os
from tqdm.notebook import tqdm

def draw_bbox(frame, bbox, color, thickness, label=''):
    x, y, w, h = [int(v) for v in bbox]
    x2, y2 = x + w, y + h
    cv2.rectangle(frame, (x, y), (x2, y2), color, thickness)
    cl = max(8, min(w, h) // 5)
    for px, py, dx, dy in [(x,y,1,1),(x2,y,-1,1),(x,y2,1,-1),(x2,y2,-1,-1)]:
        cv2.line(frame, (px, py), (px+dx*cl, py), (255,220,0), thickness+1)
        cv2.line(frame, (px, py), (px, py+dy*cl), (255,220,0), thickness+1)
    if label:
        font, fs, ft = cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2
        (tw, th), _ = cv2.getTextSize(label, font, fs, ft)
        ly = max(y - 4, th + 8)
        cv2.rectangle(frame, (x, ly-th-6), (x+tw+6, ly+2), color, -1)
        cv2.putText(frame, label, (x+3, ly-2), font, fs, (0,0,0), ft)
    return frame

def draw_trail(frame, history, length):
    pts = [(int(b[0]+b[2]/2), int(b[1]+b[3]/2)) for b in history[-length:]]
    for i in range(1, len(pts)):
        a = i / len(pts)
        cv2.line(frame, pts[i-1], pts[i], (int(50*(1-a)), int(220*a), int(255*(1-a))), 2)
    if pts:
        cv2.circle(frame, pts[-1], 4, (0,255,255), -1)

# ── Video açılışı ────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f'Video tapılmadı: {VIDEO_PATH}\nSol panel → Files → Upload edin.')

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f'📹 {width}x{height}px  |  {fps:.0f} FPS  |  {total_frames} frame  |  {total_frames/fps:.1f}s\n')

out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

ret, first_frame = cap.read()
if not ret:
    raise RuntimeError('Video oxuna bilmədi!')

tracker.initialize(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB), {'init_bbox': INIT_BBOX})
print('Tracker initialize edildi, izləmə başlayır...\n')

f0 = first_frame.copy()
draw_bbox(f0, INIT_BBOX, BOX_COLOR, BOX_THICKNESS, 'DRONE [INIT]')
cv2.putText(f0, 'OSTrack  |  Frame 0', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2)
out.write(f0)

bbox_history = [list(INIT_BBOX)]
frame_idx = 1
for frame_idx in tqdm(range(1, total_frames), desc='Track', unit='frm'):
    ret, frame = cap.read()
    if not ret:
        break
    pred_bbox = tracker.track(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))['target_bbox']
    bbox_history.append(list(pred_bbox))
    f = frame.copy()
    if SHOW_TRAIL and len(bbox_history) > 1:
        draw_trail(f, bbox_history, TRAIL_LENGTH)
    draw_bbox(f, pred_bbox, BOX_COLOR, BOX_THICKNESS, f'DRONE  #{frame_idx:04d}')
    cx, cy = int(pred_bbox[0]+pred_bbox[2]/2), int(pred_bbox[1]+pred_bbox[3]/2)
    cv2.putText(f, f'OSTrack  |  Frame {frame_idx}/{total_frames}  |  ({cx},{cy})',
                (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
    out.write(f)

cap.release()
out.release()
print(f'✅ İzləmə tamamlandı!  {frame_idx} frame  |  {os.path.getsize(OUTPUT_PATH)/1e6:.1f} MB  →  {OUTPUT_PATH}')


📹 1280x720px  |  30 FPS  |  974 frame  |  32.5s

Tracker initialize edildi, izləmə başlayır...



Track:   0%|          | 0/973 [00:00<?, ?frm/s]

✅ İzləmə tamamlandı!  971 frame  |  44.4 MB  →  /content/tracked_output.mp4


In [ ]:
# 🎞️ 7. H.264 Kodlaşdırma
OUTPUT_H264 = OUTPUT_PATH.replace('.mp4', '_h264.mp4')

print('H.264 encode başlayır...')
ret = os.system(
    f'ffmpeg -y -i {OUTPUT_PATH} -c:v libx264 -crf 20 -preset fast '
    f'-movflags +faststart {OUTPUT_H264} -loglevel warning'
)

if ret == 0 and os.path.exists(OUTPUT_H264):
    print(f'✅ H.264 hazırdır: {OUTPUT_H264}  ({os.path.getsize(OUTPUT_H264)/1e6:.1f} MB)')
else:
    print('FFmpeg uğursuz oldu — orijinal fayl istifadə ediləcək.')
    OUTPUT_H264 = OUTPUT_PATH


H.264 encode başlayır...
✅ H.264 hazırdır: /content/tracked_output_h264.mp4  (23.8 MB)


In [ ]:
# 📥 8. Nəticəni Yüklə
from google.colab import files

download_path = OUTPUT_H264 if os.path.exists(OUTPUT_H264) else OUTPUT_PATH
print(f'📥 {download_path}  ({os.path.getsize(download_path)/1e6:.1f} MB)')
files.download(download_path)
print('✅ Yükləmə başladı!')


📥 /content/tracked_output_h264.mp4  (23.8 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Yükləmə başladı!
